In [12]:
import pandas as pd
import numpy as np

def process_transport_data(stops_path, routes_path, trips_path, stop_times_path):
    print("Dosyalar okunuyor ve türler sabitleniyor...")
    
    dtype_dict = {
        'route_id': str,
        'trip_id': str,
        'stop_id': str
        # stop_sequence int olarak kalabilir, sıralama için lazım
    }
    
    # Encoding seçenekleri
    encodings = ['windows-1254', 'utf-8', 'iso-8859-9']
    
    def read_safe(path, enc_list):
        for enc in enc_list:
            try:
                return pd.read_csv(path, encoding=enc, dtype=dtype_dict, low_memory=False)
            except UnicodeDecodeError:
                continue
        raise ValueError(f"Dosya okunamadı: {path}")

    stops = read_safe(stops_path, encodings)
    routes = read_safe(routes_path, encodings)
    trips = read_safe(trips_path, encodings)
    stop_times = read_safe(stop_times_path, encodings)

    # 2. TEMİZLİK (WHITESPACE REMOVAL)
    # Bazen "28195 " ile "28195" eşleşmez. Boşlukları siliyoruz.
    for df in [stops, routes, trips, stop_times]:
        for col in ['route_id', 'trip_id', 'stop_id']:
            if col in df.columns:
                df[col] = df[col].astype(str).str.strip()

    print("Veriler birleştiriliyor...")
    
    # Adım A: Her rota için örnek BİR sefer al
    # route_id string olduğu için groupby hatasız çalışır.
    representative_trips = trips.groupby('route_id').first().reset_index()
    
    # Gereksiz sütunları atıp sadece lazım olanları alalım
    representative_trips = representative_trips[['route_id', 'trip_id', 'trip_headsign']]
    
    # Adım B: MERGE İŞLEMİ (Routes + Trips)
    # Artık iki tarafta da route_id kesinlikle STRING (Object).
    route_trips = pd.merge(
        routes[['route_id', 'route_short_name', 'route_long_name']], 
        representative_trips, 
        on='route_id', 
        how='inner' # Sadece eşleşenleri al
    )
    
    # Adım C: MERGE (Seferler + Durak Zamanları)
    trip_details = pd.merge(route_trips, stop_times[['trip_id', 'stop_id', 'stop_sequence']], on='trip_id', how='inner')
    
    # Adım D: MERGE (Durak Detayları + Koordinatlar)
    full_df = pd.merge(trip_details, stops[['stop_id', 'stop_name', 'stop_lat', 'stop_lon']], on='stop_id', how='inner')

    # 3. SIRALAMA VE HESAPLAMA
    print("Sıralama yapılıyor...")
    full_df = full_df.sort_values(by=['route_id', 'stop_sequence'])

    print("Mesafeler hesaplanıyor...")
    
    # Shift ile bir sonraki durağı al
    full_df['next_stop_name'] = full_df.groupby('route_id')['stop_name'].shift(-1)
    full_df['next_lat'] = full_df.groupby('route_id')['stop_lat'].shift(-1)
    full_df['next_lon'] = full_df.groupby('route_id')['stop_lon'].shift(-1)
    
    # Haversine
    def haversine_vec(lat1, lon1, lat2, lon2):
        R = 6371000 
        phi1, phi2 = np.radians(lat1), np.radians(lat2)
        dphi = np.radians(lat2 - lat1)
        dlambda = np.radians(lon2 - lon1)
        a = np.sin(dphi/2)**2 + np.cos(phi1)*np.cos(phi2)*np.sin(dlambda/2)**2
        c = 2 * np.arctan2(np.sqrt(a), np.sqrt(1-a))
        return R * c

    full_df['dist_meters'] = haversine_vec(
        full_df['stop_lat'].values, full_df['stop_lon'].values, 
        full_df['next_lat'].values, full_df['next_lon'].values
    )

    # Süre Hesabı
    # route_short_name sütununu string yapıp kontrol ediyoruz
    full_df['route_short_name'] = full_df['route_short_name'].astype(str)
    full_df['avg_speed_kmh'] = full_df['route_short_name'].apply(lambda x: 40 if x.startswith('34') else 25)
    
    full_df['duration_min'] = (full_df['dist_meters'] / (full_df['avg_speed_kmh'] / 3.6)) / 60
    
    # NaN olan son durakları temizle
    full_df = full_df.dropna(subset=['dist_meters'])

    return full_df[['route_short_name', 'stop_name', 'next_stop_name', 'dist_meters', 'duration_min']]

# --- ÇALIŞTIRMA ---
sonuc = process_transport_data('stops.csv', 'routes.csv', 'trips.csv', 'stop_times.csv')
print(sonuc.head())
sonuc.to_csv('final_routes.csv', index=False)

Dosyalar okunuyor ve türler sabitleniyor...
Veriler birleştiriliyor...
Sıralama yapılıyor...
Mesafeler hesaplanıyor...
    route_short_name       stop_name      next_stop_name  dist_meters  \
535               T4  Mescid-i Selam              Cebeci   729.737092   
534               T4          Cebeci      Sultançiftliği   671.904142   
533               T4  Sultançiftliği         Yenimahalle   743.218928   
532               T4     Yenimahalle          Hacı Şükrü   779.134444   
531               T4      Hacı Şükrü  50. Yıl - Baştabya   582.731322   

     duration_min  
535      1.751369  
534      1.612570  
533      1.783725  
532      1.869923  
531      1.398555  


In [10]:
# 1. Önce veriyi 'df' değişkenine atayalım (Eğer 'sonuc' hafızadaysa)
try:
    df = sonuc.copy()
except NameError:
    # Eğer kernel'i yeniden başlattıysanız ve 'sonuc' yoksa dosyadan okuyun:
    df = pd.read_csv('processed_routes.csv') # veya 'ulasim_verileri.csv'

# 2. Yenikapı ve Taksim için hesaplama fonksiyonu
def get_travel_estimate(start_stop, end_stop):
    # Başlangıç durağını bul
    start_rows = df[df['stop_name'] == start_stop]
    if start_rows.empty:
        return f"Hata: {start_stop} durağı bulunamadı."
    
    # Bitiş durağını bul
    end_rows = df[df['stop_name'] == end_stop]
    if end_rows.empty:
        return f"Hata: {end_stop} durağı bulunamadı."

    # Ortak Rota (Hat) Var mı?
    # Hem başlangıç hem bitiş durağının bulunduğu hatları (route_short_name) buluyoruz
    common_routes = set(start_rows['route_short_name']).intersection(set(end_rows['route_short_name']))
    
    if not common_routes:
        return "Bu iki durak arasında doğrudan giden tek bir hat bulunamadı."
    
    # İlk bulunan ortak hattı seçelim (Örn: M2)
    target_route = list(common_routes)[0]
    
    # O hattın verilerini çek
    route_data = df[df['route_short_name'] == target_route].reset_index(drop=True)
    
    # İndeksleri bul
    try:
        # Hatta birden fazla aynı isimli durak olabilir, ilkini alıyoruz
        start_idx = route_data[route_data['stop_name'] == start_stop].index[0]
        end_idx = route_data[route_data['stop_name'] == end_stop].index[0]
        
        # Yön Kontrolü
        if start_idx > end_idx:
            # Eğer duraklar ters sıradaysa, o hattın diğer yönünü (veya ters kesitini) bulmak lazım
            # Pratik çözüm: Veriyi ters çevirip hesaplayalım (veya ters yön verisi varsa onu kullanalım)
            # Şimdilik mutlak değer alarak mesafeyi bulabiliriz ama durakları tersten sayarız
            segments = route_data.iloc[end_idx:start_idx]
            # Ters yönde next_stop mantığı değişeceği için bu basit bir tahmin olur
            # Doğrusu: Veri setinde diğer yönün (Direction 1) olmasıdır.
            note = "(Not: Duraklar ters yönde bulundu, mesafe yaklaşık hesaplandı)"
        else:
            segments = route_data.iloc[start_idx:end_idx]
            note = ""

        total_dist = segments['dist_meters'].sum()
        total_time = segments['duration_min'].sum()
        
        return {
            "Hat": target_route,
            "Mesafe": f"{total_dist/1000:.2f} km",
            "Tahmini Süre": f"{total_time:.1f} dk",
            "Durak Sayısı": len(segments),
            "Not": note
        }
        
    except IndexError:
        return "Rota hesaplama hatası."

# --- HESAPLA ---
print(get_travel_estimate("Yenikapı", "Taksim"))

Bu iki durak arasında doğrudan giden tek bir hat bulunamadı.


In [11]:
def find_routes_flexible(start_name, end_name):
    print(f"🔍 '{start_name}' ve '{end_name}' arasındaki tüm alternatifler aranıyor...")
    
    # 1. İsim benzerliği olan TÜM durakları bul
    start_stops = df[df['stop_name'].str.contains(start_name, case=False, na=False)]
    end_stops = df[df['stop_name'].str.contains(end_name, case=False, na=False)]
    
    if start_stops.empty: return f"❌ Başlangıç durağı ({start_name}) bulunamadı."
    if end_stops.empty: return f"❌ Bitiş durağı ({end_name}) bulunamadı."

    # 2. Bu duraklardan geçen tüm hatları listele
    start_routes = set(start_stops['route_short_name'].unique())
    end_routes = set(end_stops['route_short_name'].unique())

    # 3. İkisinin kesişimini al (ORTAK HATLAR)
    common_routes = start_routes.intersection(end_routes)
    
    if not common_routes:
        return "⚠️ Doğrudan giden tek bir hat bulunamadı (Aktarma gerekebilir)."

    results = []
    
    # 4. Her ortak hat için hesaplama yap
    for route in common_routes:
        # O hattın verisini çek
        route_df = df[df['route_short_name'] == route].sort_values(by='stop_sequence')
        
        # Bu hatta, bizim aradığımız durak isimlerine uyan satırları bul
        # (Start ve End duraklarının bu hattaki tam karşılığını buluyoruz)
        s_matches = route_df[route_df['stop_name'].str.contains(start_name, case=False, na=False)]
        e_matches = route_df[route_df['stop_name'].str.contains(end_name, case=False, na=False)]
        
        if s_matches.empty or e_matches.empty:
            continue
            
        # İlk eşleşmeleri al (Genelde doğru yöndür)
        s_idx = s_matches.index[0]
        e_idx = e_matches.index[0]
        
        # Durakların sırasını kontrol et
        # Eğer start index > end index ise hat ters yöne gidiyordur veya biz tersten bakıyoruz
        if s_idx > e_idx:
            # Ters yön hesabı (basitçe aradaki segmentleri alıyoruz)
            segment = route_df.loc[e_idx:s_idx] 
            note = "(Dönüş Yönü Olabilir)"
        else:
            # Düz yön
            segment = route_df.loc[s_idx:e_idx]
            note = ""
            
        # Mesafeyi ve süreyi topla
        dist = segment['dist_meters'].sum() / 1000 # km
        dur = segment['duration_min'].sum() # dk
        stops_count = len(segment)
        
        results.append({
            "Hat": route,
            "Güzergah": f"{segment.iloc[0]['stop_name']} -> {segment.iloc[-1]['stop_name']}",
            "Mesafe": f"{dist:.2f} km",
            "Süre": f"{dur:.1f} dk",
            "Durak Sayısı": stops_count,
            "Not": note
        })

    return pd.DataFrame(results)

# --- TEST ET ---
# Artık Yenikapı - Taksim dediğinde M2'yi bulması lazım
alternatifler = find_routes_flexible("Yenikapı", "Taksim")
print(alternatifler)

🔍 'Yenikapı' ve 'Taksim' arasındaki tüm alternatifler aranıyor...
⚠️ Doğrudan giden tek bir hat bulunamadı (Aktarma gerekebilir).


In [13]:
# Daha önce tanımladığımız 'find_routes_flexible' fonksiyonunu kullanıyoruz
# İsimleri genel aratıyoruz ki "Cevizlibağ - AÖY" gibi varyasyonları da bulsun.

print("--- Metrobüs Hattı Analizi (Cevizlibağ - Beylikdüzü) ---")
sonuclar = find_routes_flexible("Cevizlibağ", "Beylikdüzü")

# Sonuçları daha okunaklı gösterelim
if isinstance(sonuclar, pd.DataFrame) and not sonuclar.empty:
    # Sadece en mantıklı (mesafesi 0 olmayan) sonuçları göster
    temiz_sonuc = sonuclar[sonuclar['Mesafe'] != '0.00 km']
    print(temiz_sonuc.to_string(index=False))
else:
    print(sonuclar)

--- Metrobüs Hattı Analizi (Cevizlibağ - Beylikdüzü) ---
🔍 'Cevizlibağ' ve 'Beylikdüzü' arasındaki tüm alternatifler aranıyor...
⚠️ Doğrudan giden tek bir hat bulunamadı (Aktarma gerekebilir).


In [14]:
def durak_analizi(durak_adi):
    print(f"--- '{durak_adi}' İÇİN HAT ANALİZİ ---")
    
    # 1. İsim eşleşen durakları bul
    eslesen_duraklar = df[df['stop_name'].str.contains(durak_adi, case=False, na=False)]
    
    if eslesen_duraklar.empty:
        print("❌ Hiçbir durak bulunamadı!")
        return

    # 2. Her duraktan geçen hatları listele
    for durak in eslesen_duraklar['stop_name'].unique():
        hatlar = eslesen_duraklar[eslesen_duraklar['stop_name'] == durak]['route_short_name'].unique()
        print(f"Durak: {durak}")
        print(f"  └── Geçen Hatlar: {', '.join(map(str, hatlar))}")
        print("-" * 20)

# Cevizlibağ ve Beylikdüzü'ne kimler uğruyor görelim
durak_analizi("Cevizlibağ")
durak_analizi("Beylikdüzü")

--- 'Cevizlibağ' İÇİN HAT ANALİZİ ---
Durak: Cevizlibağ
  └── Geçen Hatlar: CEVİZLİBAĞ - ADNAN MENDERES VATAN BLV- TAKSİM, CEVİZLİBAĞ - TURGUT ÖZAL MİLLET CAD-TAKSİM
--------------------
Durak: Cevizlibağ - AÖY
  └── Geçen Hatlar: T1
--------------------
--- 'Beylikdüzü' İÇİN HAT ANALİZİ ---
Durak: BEYLİKDÜZÜ
  └── Geçen Hatlar: BÜYÜKÇEKMECE-BEYKENT, BEYLİKDÜZÜ KAYMAKAMLIĞI-BÜYÜKÇEKMECE
--------------------
Durak: BEYLİKDÜZÜ BELEDİYE
  └── Geçen Hatlar: BÜYÜKÇEKMECE-BEYKENT, BEYLİKDÜZÜ BELEDİYESİ-BÜYÜKÇEKMECE ATİRUS AVM, DEREAĞZI MH-YAKUPLU HASIRCILAR İŞ MERKEZİ, DEREAĞZI MH-MİGROS AVM
--------------------
Durak: BEYLİKDÜZÜ (E5)
  └── Geçen Hatlar: BÜYÜKÇEKMECE-ZAFER MAHALLESİ, 2.GÜZERGAH, DEREAĞZI MH-YAKUPLU HASIRCILAR İŞ MERKEZİ, BÜYÜKÇEKMECE-DOĞAN ARASLI-MİMSAN
--------------------
Durak: BEYLİKDÜZÜ İSKİ
  └── Geçen Hatlar: 2.GÜZERGAH, DEREAĞZI MH-YAKUPLU HASIRCILAR İŞ MERKEZİ, BÜYÜKÇEKMECE-DOĞAN ARASLI-MİMSAN
--------------------
Durak: BEYLİKDÜZÜ İMAM HATP
  └── Geçen Hatlar: YAKU

In [17]:
def find_routes_flexible(start_name, end_name):
    print(f"🔍 '{start_name}' ve '{end_name}' arasındaki tüm alternatifler aranıyor...")
    
    # 1. İsim benzerliği olan TÜM durakları bul
    start_stops = df[df['stop_name'].str.contains(start_name, case=False, na=False)]
    end_stops = df[df['stop_name'].str.contains(end_name, case=False, na=False)]
    
    if start_stops.empty: return f"❌ Başlangıç durağı ({start_name}) bulunamadı."
    if end_stops.empty: return f"❌ Bitiş durağı ({end_name}) bulunamadı."

    # 2. Bu duraklardan geçen tüm hatları listele
    start_routes = set(start_stops['route_short_name'].unique())
    end_routes = set(end_stops['route_short_name'].unique())

    # 3. İkisinin kesişimini al (ORTAK HATLAR)
    common_routes = start_routes.intersection(end_routes)
    
    if not common_routes:
        return "⚠️ Doğrudan giden tek bir hat bulunamadı (Aktarma gerekebilir)."

    results = []
    
    # 4. Her ortak hat için hesaplama yap
    for route in common_routes:
        # HATA ÇÖZÜMÜ: .sort_values(by='stop_sequence') KOMUTU KALDIRILDI.
        # Çünkü verilerimiz zaten sıralı ve 'stop_sequence' sütunu artık yok.
        route_df = df[df['route_short_name'] == route]
        
        # Bu hatta, bizim aradığımız durak isimlerine uyan satırları bul
        s_matches = route_df[route_df['stop_name'].str.contains(start_name, case=False, na=False)]
        e_matches = route_df[route_df['stop_name'].str.contains(end_name, case=False, na=False)]
        
        if s_matches.empty or e_matches.empty:
            continue
            
        # İlk eşleşmeleri al
        s_idx = s_matches.index[0]
        e_idx = e_matches.index[0]
        
        # Yön Kontrolü ve Segment Seçimi
        if s_idx > e_idx:
            # Ters yön
            segment = df.loc[e_idx:s_idx] # Ana df üzerinden slicing yapıyoruz
            note = "(Ters Yön / Dönüş)"
        else:
            # Düz yön
            segment = df.loc[s_idx:e_idx]
            note = ""
            
        # Mesafeyi ve süreyi topla
        dist = segment['dist_meters'].sum() / 1000 # km
        dur = segment['duration_min'].sum() # dk
        stops_count = len(segment)
        
        results.append({
            "Hat": route,
            "Güzergah": f"{segment.iloc[0]['stop_name']} -> {segment.iloc[-1]['stop_name']}",
            "Mesafe": f"{dist:.2f} km",
            "Süre": f"{dur:.1f} dk",
            "Durak Sayısı": stops_count,
            "Not": note
        })

    return pd.DataFrame(results)

# --- TEST ET ---
# Şimdi M2 (Yenikapı - Taksim) sorunsuz çalışmalı
print("\n🔄 M2 Rotası Tekrar Test Ediliyor...")
sonuc = find_routes_flexible("Yenikapı", "Taksim")
print(sonuc)


🔄 M2 Rotası Tekrar Test Ediliyor...
🔍 'Yenikapı' ve 'Taksim' arasındaki tüm alternatifler aranıyor...
  Hat            Güzergah      Mesafe        Süre  Durak Sayısı  \
0  M2  Taksim -> Yenikapı  5961.39 km  14264.5 dk         10901   

                  Not  
0  (Ters Yön / Dönüş)  
